# Real-World Data Representation Using Tensors: Images, Volumetric Data, Tables, Time Series, and Text

<a href="https://colab.research.google.com/github/emreaslan7/ai/blob/main/notebooks/deep-learning-with-pytorch/04-real-world-data-representation-using-tensors.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This notebook demonstrates the end-to-end transformation of raw physical data modalities (2D images, 3D volumetric medical scans, tabular spreadsheets, cyclical time series, and natural language text) into continuous PyTorch tensors.

## 0. Setup and Environment
Verify GPU availability, set random seeds, and import necessary libraries.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import imageio.v2 as imageio
import io
import urllib.request
import re

# Set deterministic seed and verify device
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch Version: {torch.__version__} | Active Device: {device}")

## 1. Working with Images (2D Data)
Loading RGB images, converting $H \times W \times C$ to $C \times H \times W$ layout, batching, and per-channel standardization.

In [ ]:
# 1.1 Load sample image (or synthetic placeholder)
img_url = 'https://raw.githubusercontent.com/deep-learning-with-pytorch/dlwpt-code/master/data/p1ch4/image-dog/bobby.jpg'
try:
    req = urllib.request.Request(img_url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=5) as resp:
        img_arr = imageio.imread(resp.read())
except Exception as e:
    print(f"Using synthetic image array (Fallback: {e})")
    img_arr = np.random.randint(0, 256, (720, 1280, 3), dtype=np.uint8)

print("NumPy image array shape (H, W, C):", img_arr.shape)
print("Image dtype:", img_arr.dtype)

In [ ]:
# 1.2 Convert to Tensor and Permute dimensions (H, W, C) -> (C, H, W)
img_tensor = torch.from_numpy(img_arr)
img_chw = img_tensor.permute(2, 0, 1)

print("Permuted PyTorch Tensor shape (C, H, W):", img_chw.shape)
print("Strides:", img_chw.stride())
print("Is Contiguous in Memory:", img_chw.is_contiguous())

In [ ]:
# 1.3 Pre-allocate a 4D Batch Tensor and populate slices
batch_size, channels, height, width = 3, 3, 256, 256
batch = torch.zeros(batch_size, channels, height, width, dtype=torch.uint8)

for i in range(batch_size):
    dummy_img = torch.randint(0, 256, (height, width, channels), dtype=torch.uint8)
    batch[i] = dummy_img.permute(2, 0, 1)

print("Pre-allocated 4D Batch Tensor shape (N, C, H, W):", batch.shape)

In [ ]:
# 1.4 Scale to [0.0, 1.0] and compute per-channel Z-score standardization
batch_float = batch.float() / 255.0

mean = batch_float.mean(dim=[0, 2, 3])
std = batch_float.std(dim=[0, 2, 3])

batch_normalized = (batch_float - mean.view(1, 3, 1, 1)) / std.view(1, 3, 1, 1)

print("Per-channel Mean (R, G, B):", mean)
print("Per-channel Std  (R, G, B):", std)
print("Normalized Batch Shape:", batch_normalized.shape)
print("Normalized Channel 0 Mean:", batch_normalized[:, 0].mean().item())
print("Normalized Channel 0 Std: ", batch_normalized[:, 0].std().item())

## 2. 3D Images: Volumetric Medical Data
Representing 3D CT/MRI scans as 5D tensors $(N, C, D, H, W)$ with Hounsfield Unit (HU) windowing.

In [ ]:
# 2.1 Simulate 3D CT scan volume (Depth x Height x Width)
vol_depth, vol_height, vol_width = 64, 256, 256
raw_ct_volume = torch.randint(-1000, 1500, (vol_depth, vol_height, vol_width), dtype=torch.float32)

# Expand to 5D Volumetric Tensor: (N, C, D, H, W)
vol_5d = raw_ct_volume.unsqueeze(0).unsqueeze(0)
print("5D Volumetric Batch Shape (N, C, D, H, W):", vol_5d.shape)

# 2.2 Clinical Radiodensity Windowing for Lung Tissue [-1000 HU, +400 HU]
lung_min, lung_max = -1000.0, 400.0
vol_clipped = torch.clamp(vol_5d, min=lung_min, max=lung_max)
vol_normalized = (vol_clipped - lung_min) / (lung_max - lung_min)

print("Normalized Lung CT Volume Range:", vol_normalized.min().item(), "to", vol_normalized.max().item())

## 3. Representing Tabular Data (UCI Wine Quality)
Loading heterogeneous tabular columns, continuous vs categorical targets, One-Hot Encoding, and $Z$-score standardization.

In [ ]:
# 3.1 Load UCI Wine Quality dataset
wine_url = 'https://raw.githubusercontent.com/deep-learning-with-pytorch/dlwpt-code/master/data/p1ch4/tabular-wine/winequality-white.csv'
try:
    req = urllib.request.Request(wine_url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=5) as resp:
        csv_text = resp.read().decode('utf-8')
    wineq_numpy = np.loadtxt(io.StringIO(csv_text), dtype=np.float32, delimiter=';', skiprows=1)
except Exception as e:
    print(f"Generating synthetic wine tabular matrix (Fallback: {e})")
    wineq_numpy = np.random.randn(4898, 12).astype(np.float32)
    wineq_numpy[:, -1] = np.random.randint(3, 9, size=4898)

wineq = torch.from_numpy(wineq_numpy)
print("Loaded Wine Table Tensor shape (Samples, Columns):", wineq.shape)
print("Tensor dtype:", wineq.dtype)

In [ ]:
# 3.2 Split features (X) and target quality ratings (y)
data = wineq[:, :-1]
target = wineq[:, -1].long()

print("Input Features Shape (N, D):", data.shape)
print("Target Scores Shape (N):", target.shape)
print("Sample Target Scores:", target[:5])

In [ ]:
# 3.3 One-Hot Encoding: Low-level scatter_ vs Modern F.one_hot
num_classes = 10

# Method 1: scatter_
target_onehot_scatter = torch.zeros(target.shape[0], num_classes).scatter_(1, target.unsqueeze(1), 1.0)

# Method 2: F.one_hot
target_onehot_fn = F.one_hot(target, num_classes=num_classes).float()

assert torch.equal(target_onehot_scatter, target_onehot_fn)
print("One-Hot Encoded Target Shape:", target_onehot_fn.shape)
print("Sample One-Hot vector for score", target[0].item(), ":
", target_onehot_fn[0])

In [ ]:
# 3.4 Column-wise Z-Score Standardization
data_mean = torch.mean(data, dim=0)
data_var = torch.var(data, dim=0, unbiased=False)

data_normalized = (data - data_mean) / torch.sqrt(data_var + 1e-7)

print("Normalized Data Shape:", data_normalized.shape)
print("Column 0 Mean:", data_normalized[:, 0].mean().item())
print("Column 0 Std: ", data_normalized[:, 0].std().item())

In [ ]:
# 3.5 Rule-Based Threshold Classification (Good > 5 vs Bad <= 5)
bad_indexes = target <= 5
good_indexes = target > 5

total_sulfur_threshold = 141.83
predicted_good = data[:, 6] < total_sulfur_threshold
actual_good = target > 5

accuracy = (predicted_good == actual_good).float().mean().item()
print(f"Rule-Based Accuracy on Sulfur Dioxide Threshold: {accuracy * 100:.2f}%")

## 4. Working with Time Series Data (Capital Bikeshare)
Folding 2D temporal logs into 3D $(N, L, C)$ tensors, layout transpositions, and cyclical feature concatenation.

In [ ]:
# 4.1 Load Capital Bikeshare hourly dataset
bike_url = 'https://raw.githubusercontent.com/deep-learning-with-pytorch/dlwpt-code/master/data/p1ch4/bike-sharing-dataset/hour-fixed.csv'
try:
    req = urllib.request.Request(bike_url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=5) as resp:
        csv_bytes = resp.read()
    bikes_numpy = np.loadtxt(io.BytesIO(csv_bytes), dtype=np.float32, delimiter=',', skiprows=1,
                             converters={1: lambda s: float(s[8:10])})
except Exception as e:
    print(f"Generating synthetic bikeshare logs (Fallback: {e})")
    bikes_numpy = np.random.randn(17520, 17).astype(np.float32)
    bikes_numpy[:, 9] = np.random.randint(1, 5, size=17520)

bikes = torch.from_numpy(bikes_numpy)
print("Raw 2D Hourly Log Shape (Total Hours, Features):", bikes.shape)

In [ ]:
# 4.2 Reshape 2D tensor into 3D daily sequences (730 Days x 24 Hours x 17 Features)
daily_bikes = bikes.view(-1, 24, bikes.shape[1])
print("3D Temporal Tensor Shape (N, L, C):", daily_bikes.shape)

# Transpose to Channels-First Sequence (N, C, L) for 1D convolutions
daily_bikes_ncl = daily_bikes.transpose(1, 2)
print("Channels-First Sequence Shape (N, C, L):", daily_bikes_ncl.shape)

In [ ]:
# 4.3 One-Hot Encode Weather Condition (Column 9) and Concatenate along Channels
weather_classes = (daily_bikes[:, :, 9].long() - 1).clamp(0, 3)
weather_onehot = F.one_hot(weather_classes, num_classes=4).float().transpose(1, 2)

bikes_augmented = torch.cat([weather_onehot, daily_bikes_ncl], dim=1)
print("Augmented Multimodal Tensor Shape (N, C_total, L):", bikes_augmented.shape)

## 5. Representing Text (Character & Word Tokens to Dense Embeddings)
Comparing One-Hot sparse token matrices with continuous dense vector embeddings via `nn.Embedding`.

In [ ]:
# 5.1 Character-Level One-Hot Matrix
sample_text = 'Deep Learning with PyTorch'
vocab_size = 128

char_onehot = torch.zeros(len(sample_text), vocab_size)
for i, char in enumerate(sample_text):
    if ord(char) < vocab_size:
        char_onehot[i, ord(char)] = 1.0

print("Character One-Hot Tensor Shape (Seq_Len, Vocab_Size):", char_onehot.shape)

In [ ]:
# 5.2 Word-Level Tokenization and Dense Continuous Embedding (nn.Embedding)
sentence = 'Deep learning models project discrete symbolic tokens into continuous latent manifolds'
words = re.findall(r'\w+', sentence.lower())

word_to_idx = {w: i for i, w in enumerate(sorted(set(words)))}
token_ids = torch.tensor([word_to_idx[w] for w in words], dtype=torch.long)

# Continuous Embedding Layer
embed_dim = 64
embedding_layer = nn.Embedding(num_embeddings=len(word_to_idx), embedding_dim=embed_dim)
dense_embeddings = embedding_layer(token_ids)

print("Input Token IDs:", token_ids)
print("Dense Embedding Matrix Shape (Seq_Len, Embed_Dim):", dense_embeddings.shape)

## 6. Chapter Exercises Solutions
Rolling strided windows on temporal series and Python source code tokenization.

In [ ]:
# Exercise 2: Rolling strided temporal windows via Tensor.unfold()
time_series = torch.randn(100, 5)  # 100 hours, 5 features
window_size, step_size = 24, 1

rolling_windows = time_series.unfold(dimension=0, size=window_size, step=step_size).permute(0, 2, 1)
print("Extracted Rolling Windows Shape (N_windows, Window_Len, Features):", rolling_windows.shape)

In [ ]:
# Exercise 3: Character tokenization of Python source code with nn.Embedding
code = 'def forward(self, x): return self.linear(x)'
unique_chars = sorted(list(set(code)))
c2i = {c: i for i, c in enumerate(unique_chars)}

code_ids = torch.tensor([c2i[c] for c in code], dtype=torch.long)
code_embed = nn.Embedding(len(c2i), 32)(code_ids)
print(f"Code Characters: {len(code)} | Unique: {len(c2i)} | Embedded Shape: {code_embed.shape}")